In [1]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

from matplotlib.backends.backend_pdf import PdfPages
import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        #print('Covariance matrix \n%s' %covariance_matrix)

        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]
        #print('Eigenvectors \n%s' %neww)
        #print('\nEigenvalues \n%s' %sortedd)
    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCAnew(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    
    indexA=zx2['Index']
    
    #Empty Dataframe
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,

#---------------------------------------------------------------------------------------------------------------

In [3]:
#MM waves folder

#MM_folder="C:/Users/Ariel/Desktop/JUPYTER/Mirror_Mode_Waves_Linear_Regression/DATA"
MM_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED"
#---------------------------------------------------------

list_of_files_MM=get_directory(MM_folder)

#List with the paths
newlist2=[]
for item in list_of_files_MM:
    newlist2.append(MM_folder+'/'+str(item))
print(newlist2)
events=[] #Amount of MM waves per day
for i in np.arange(0, len(newlist2)):
        title2=['Days','Normalized_Data','Data_Plasma','Waves']
        path= str(newlist2[i])
        data1= pd.read_table(path, header=None, names=title2, sep='\t', engine='python')
        #data1= pd.read_table(path)
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        #data2=data2.reset_index(drop=True)
        events.append(data2)


['C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201411', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201412', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201501', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201502', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201503', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201504', 'C:/Users/Ariel

In [12]:
print(events[2])

amountofplasmadata=[]
for i in np.arange(0,len(events)):
    counter=0
    for j in np.arange(1,len(events[i])+1):
        counter=counter+int(events[i]['Data_Plasma'][j])
    amountofplasmadata.append(counter)    
        
print(amountofplasmadata)    

   Days Normalized_Data Data_Plasma Waves
1     1             NaN           0     0
2     2             NaN           0     0
3     3             NaN           0     0
4     4             NaN           0     0
5     5             NaN           0     0
6     6             NaN           0     0
7     7             NaN           0     0
8     8             NaN           0     0
9     9             NaN           0     0
10   10             NaN           0     0
11   11             NaN           0     0
12   12             NaN           0     0
13   13             NaN           0     0
14   14             NaN           0     0
15   15             NaN           0     0
16   16             NaN           0     0
17   17             NaN           0     0
18   18             NaN           0     0
19   19             NaN           0     0
20   20             NaN           0     0
21   21             NaN           0     0
22   22             NaN           0     0
23   23             NaN           